# Chapter 13 — How Do You Evaluate an Embedding?

**Book alignment:** Embeddings From First Principles, Chapter 13

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** A leaderboard number is a population statistic; you
have one task. Does changing the *query difficulty* reorder the models? On RELATE, v0.1's
near-restatement queries pin three encoders at ~0.94 nDCG@10 with `mpnet` on top; v0.2's
142 hard queries drop them to ~0.85 and put `bge-large` on top (the committed Wave 1
measurement).

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Metrics fit consumers — a 3-line reminder

In [ ]:
# ranked list with graded relevance; two consumers, two metrics
ranked = [(0, 3), (1, 0), (2, 2), (3, 0), (4, 1)]      # (doc, relevance)
def recall_at_1(r): return 1.0 if r[0][1] >= 3 else 0.0
def mrr(r):
    for i, (_, rel) in enumerate(r, 1):
        if rel >= 3:
            return 1 / i
    return 0.0
print("RAG (reads the first hit)  -> MRR / Recall@1:", mrr(ranked), recall_at_1(ranked))
print("aggregator (needs coverage) -> would weight Recall@k / MAP instead")

## 2. Query difficulty reorders models (Wave 1: v0.1 vs v0.2)

In [ ]:
v01 = art("wave1", "relevance-definition-sweep.json")["models"]
v02 = art("wave1-v02", "relevance-definition-sweep.json")["models"]

def winner(models, key="answers_only(>=3)"):
    return max(models, key=lambda m: models[m][key]["ndcg10"])

print("RELATE v0.1 (near-restatement queries):")
for m in v01:
    print(f"  {m:12} nDCG@10 = {v01[m]['answers_only(>=3)']['ndcg10']:.4f}")
print("  winner:", winner(v01))
print("\nRELATE v0.2 (142 hard queries, real lexical gap):")
for m in v02:
    print(f"  {m:12} nDCG@10 = {v02[m]['answers_only(>=3)']['ndcg10']:.4f}")
print("  winner:", winner(v02))

assert winner(v01) == "mpnet-base"
assert winner(v02) == "bge-large"
# v0.1 saturates near 0.94; v0.2 separates the models near 0.85
assert min(m["answers_only(>=3)"]["ndcg10"] for m in v01.values()) > 0.93
assert max(m["answers_only(>=3)"]["ndcg10"] for m in v02.values()) < 0.89
print("\nthe query set was the ceiling; a private eval as hard as your product's is what reorders models")

## What we earned

Representation quality (a multi-task leaderboard) is portable and comparative; application
quality is yours — opened by domain, relevance definition, query style, consumer, and
asymmetry. On RELATE, easy queries pin every model within 0.02 nDCG@10 with `mpnet` nominally
on top; hard queries drop the field to ~0.85 and flip the winner to `bge-large`. Model
selection needs a private eval whose queries are as hard as production's.

**Notebook 14 / Chapter 14** attacks the score itself: a cosine of 0.81 means nothing until
you know the two distributions it came from.